In [13]:
!pip install rdkit
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

In [14]:
leaving_groups = {
    "Br",
    "Cl",
    "I",
    "F"}
cpds=input("Enter the molecule: ")
mol=Chem.MolFromSmiles(cpds)
if mol is None:
    print("Invalid SMILES.")
    exit()
def find_substrate_carbon(mo):
    try:
       atoms=mo.GetAtoms()
       for atom in atoms:
        if atom.GetSymbol() in leaving_groups:
            for neighbors in  atom.GetNeighbors():
                if neighbors.GetSymbol()=="C":
                    return neighbors
    except AttributeError:
        return None
if mol is not None:
    atomos=find_substrate_carbon(mol)
def classify_carbon(atom):
    carbon_count = 0
    for a in atom.GetNeighbors():
        if a.GetSymbol()=="C":
            carbon_count=carbon_count + 1
    if carbon_count ==1:
        return "primary"
    elif carbon_count ==2:
        return "Secondary"
    elif carbon_count ==3:
        return "Tertariy"
substrate_classi=classify_carbon(atomos)
nucleophiles_data = {
    "oh":   {"nucleophilicity": "strong",   "basicity": "strong",   "bulky": False},  # hydroxide
    "cn":   {"nucleophilicity": "strong",   "basicity": "weak",     "bulky": False},  # cyanide
    "i":    {"nucleophilicity": "strong",   "basicity": "weak",     "bulky": False},  # iodide
    "br":   {"nucleophilicity": "moderate", "basicity": "weak",     "bulky": False},  # bromide
    "cl":   {"nucleophilicity": "weak",     "basicity": "moderate", "bulky": False},  # chloride
    "sh":   {"nucleophilicity": "strong",   "basicity": "weak",     "bulky": False},  # hydrosulfide
    "n3":   {"nucleophilicity": "strong",   "basicity": "weak",     "bulky": False},  # azide
    "aco":  {"nucleophilicity": "moderate", "basicity": "moderate", "bulky": False},  # acetate
    "otbu": {"nucleophilicity": "weak",     "basicity": "strong",   "bulky": True},   # tert-butoxide
    "oet":  {"nucleophilicity": "moderate", "basicity": "strong",   "bulky": False},  # ethoxide
    "nh2":  {"nucleophilicity": "strong",   "basicity": "strong",   "bulky": False},  # amide ion
    "h2o":  {"nucleophilicity": "weak",     "basicity": "weak",     "bulky": False},  # water
    "etoh": {"nucleophilicity": "weak",     "basicity": "weak",     "bulky": False},  # ethanol
    "f":    {"nucleophilicity": "weak",     "basicity": "moderate", "bulky": False},  # fluoride
    "scn":  {"nucleophilicity": "strong",   "basicity": "weak",     "bulky": False},  # thiocyanate
}
ask_Nu=input("Enter a nucleophile:*IF it is F[-] use F*").lower()
ask_temp=float(input("Enter the temperature :"))
reaction_mecanism_score={"SN2":0,"SN1":0,"E1":0,"E2":0}
if substrate_classi =="primary":
    reaction_mecanism_score["SN2"]+=2
    reaction_mecanism_score["E2"]+=2
    if ask_Nu in nucleophiles_data.keys():
        if nucleophiles_data[ask_Nu]["nucleophilicity"]=="strong":
           reaction_mecanism_score["SN2"]+=2
        if nucleophiles_data[ask_Nu]["nucleophilicity"]=="moderate":
            reaction_mecanism_score["SN2"]+=1
        if nucleophiles_data[ask_Nu]["bulky"]==True:
           reaction_mecanism_score["E2"]+=2
        if nucleophiles_data[ask_Nu]["basicity"]=="strong":
           reaction_mecanism_score["E2"]+=2
        if ask_temp > 50:
            reaction_mecanism_score["E2"]+=1
        print("score for E2",reaction_mecanism_score["E2"])
        print("score for SN2",reaction_mecanism_score["SN2"])
    if  ask_Nu not in nucleophiles_data.keys():
        print("We do not have the nucelophile or you mistyped")
import csv
with open("/content/sample_data/solution.csv","w") as file:
    s=file.write ("name,polarity,type,chem21_category,toxicity_tier\n")
    s_1=file.write("Water,high,protic,recommended,1\n")
    s_2=file.write("Methanol,high,protic,recommended,1\n")
    s_3=file.write("Ethanol,high,protic,recommended,1\n")
    s_4=file.write("Acetone,medium,aprotic,recommended,1\n")
    s_5=file.write("Isopropanol,high,protic,recommended,1\n")
    s_6=file.write("Ethyl_acetate,medium,aprotic,recommended,1\n")
    s_7=file.write("DMSO,high,aprotic,problematic,2\n")
    s_8=file.write("THF,medium,aprotic,problematic,2\n")
    s_9=file.write("DCM,medium,aprotic,problematic,2\n")
    s_10=file.write("Cyclohexane,low,aprotic,problematic,2\n")
    s_11=file.write("Formic_acid,high,protic,problematic,2\n")
    s_12=file.write("Acetonitrile,high,aprotic,problematic,2\n")
    s_13=file.write("1_4-Dioxane,medium,aprotic,problematic,2\n")
    s_14=file.write("MTBE,low,aprotic,problematic,2\n")
    s_15=file.write("DMF,high,aprotic,hazardous,3\n")
    s_16=file.write("Hexane,low,aprotic,hazardous,3\n")
    s_17=file.write("Pentane,low,aprotic,hazardous,3\n")
    s_18=file.write("Pyridine,medium,aprotic,hazardous,3\n")
    s_19=file.write("Triethylamine,low,aprotic,hazardous,3\n")
    s_20=file.write("Chloroform,medium,aprotic,hazardous,3\n")
    s_21=file.write("Benzene,low,aprotic,highly_hazardous,4\n")
    s_22=file.write("Diethyl_ether,low,aprotic,highly_hazardous,4\n")
    s_23=file.write("Nitromethane,high,aprotic,highly_hazardous,4\n")

with open("/content/sample_data/solution.csv","r") as file:
    reader=csv.DictReader(file)
    all_solvents = list(reader)
def rank_solvents_by_toxicity(solvent_list):
                   for solve in solvent_list:
                     solve["toxicity_tier"] = int(solve["toxicity_tier"])
                   ranked = sorted(solvent_list, key=lambda x: x["toxicity_tier"])
                   return ranked
ranked_solvents=rank_solvents_by_toxicity(all_solvents)
#print(ranked_solvents)
if reaction_mecanism_score["E2"] > reaction_mecanism_score["SN2"]:
    print("Solvents for E2")
    for r in ranked_solvents:
        if r["type"] == "protic":
            print(r["name"])
elif reaction_mecanism_score["E2"] < reaction_mecanism_score["SN2"]:
    print("Solvents for SN2")
    for r in ranked_solvents:
        if r["type"] == "aprotic":
            print(r["name"])
def scores_to_percentage(scores_dict):
    total = scores_dict["SN2"] + scores_dict["E2"]
    if total == 0:
        return {"SN2": 0, "E2": 0}
    sn2_pct = round((scores_dict["SN2"] / total) * 100, 1)
    e2_pct = round((scores_dict["E2"] / total) * 100, 1)
    return {"SN2": sn2_pct, "E2": e2_pct}
percentage=scores_to_percentage(reaction_mecanism_score)
print(percentage)
leaving_group_quality = {
    "I": "excellent",
    "Br": "good",
    "Cl": "moderate",
    "F": "poor",
}

def describe_leaving_group(leaving_group):
    quality = leaving_group_quality.get(leaving_group, "unknown")
    if quality == "poor":
        return "Warning: this leaving group is poor — this reaction may proceed very slowly or not at all, regardless of the predicted mechanism below."
    return f"Leaving group quality: {quality}."

def find_leaving_group_symbol(mo):
    atoms = mo.GetAtoms()
    for atom in atoms:
        if atom.GetSymbol() in leaving_groups:
            return atom.GetSymbol()
    return None

lg_symbol = find_leaving_group_symbol(mol)
if lg_symbol is not None:
    print(describe_leaving_group(lg_symbol))
else:
    print("No recognized leaving group found.")

def explain_prediction(ask_Nu, nucleophiles_data, ask_temp, reaction_mecanism_score):
    reasons = []
    nu_data = nucleophiles_data[ask_Nu]

    reasons.append("Primary substrate: backside attack is accessible, so SN2 and E2 both start as real possibilities.")

    if nu_data["nucleophilicity"] == "strong":
        reasons.append(f"'{ask_Nu}' is a strong nucleophile, favoring SN2.")
    elif nu_data["nucleophilicity"] == "moderate":
        reasons.append(f"'{ask_Nu}' is a moderate nucleophile, mildly favoring SN2.")
    else:
        reasons.append(f"'{ask_Nu}' is a weak nucleophile, giving no boost to SN2.")

    if nu_data["bulky"]:
        reasons.append(f"'{ask_Nu}' is bulky, blocking backside attack and favoring E2.")

    if nu_data["basicity"] == "strong":
        reasons.append(f"'{ask_Nu}' is a strong base, favoring E2.")

    if ask_temp > 35:
        reasons.append(f"Elevated temperature ({ask_temp}C) mildly favors E2 (entropy-driven).")

    winner = "E2" if reaction_mecanism_score["E2"] > reaction_mecanism_score["SN2"] else "SN2"
    reasons.append(f"Overall: {winner} is favored based on the combined factors above.")

    return reasons

for r in explain_prediction(ask_Nu, nucleophiles_data, ask_temp, reaction_mecanism_score):
    print("-", r)

Enter the molecule: CCBr
Enter a nucleophile:*IF it is F[-] use F*OH
Enter the temperature :45
score for E2 5
score for SN2 4
Solvents for E2
Water
Methanol
Ethanol
Isopropanol
Formic_acid
{'SN2': 44.4, 'E2': 55.6}
Leaving group quality: good.
- Primary substrate: backside attack is accessible, so SN2 and E2 both start as real possibilities.
- 'oh' is a strong nucleophile, favoring SN2.
- 'oh' is a strong base, favoring E2.
- Elevated temperature (45.0C) mildly favors E2 (entropy-driven).
- Overall: E2 is favored based on the combined factors above.
